In [1]:
import requests as req
from bs4 import BeautifulSoup
import pandas as pd
import re


source = {}

master_data = {}

def parse_sources():

    with open("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/raw/links_for_military_data.txt", "r") as file:
        lines = file.readlines()


    sources = False
    for line in lines:
        line = line.strip()
        if line.startswith("other_sources"):
            sources = True
            continue
        elif line == "}":
            break
        elif sources and line:

            line = line.rstrip(",")

            if ": " in line:
                key, value = line.split(": ", 1)

                key = key.strip().strip("'").strip('"')
                value = value.strip().strip("'").strip('"')
                source[key] = value

parse_sources()

# print(source)


In [ ]:
def get_links():

    for key,val in source.items():
        response = req.get(key)

        # file_name = "/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/raw/" + val + ".html"

        # file = open(file_name,"w+")
        # file.write(response.text)
        # file.close()

        soup = BeautifulSoup(response.text,"html.parser")

        rec = soup.find_all("div",class_="recordsetContainer")

        for r in rec:

             cnt_shrtname = r.select_one("div.shortFormName span").get_text(strip=True)
             cnt_longname = r.select_one("div.longFormName span").get_text(strip=True)
             cnt_val = r.select_one("div.valueContainer span span").get_text(strip=True)
             cnt_val = re.sub(r"[^0-9]","",cnt_val)
             cnt_val = "".join(cnt_val.split())
             cnt_val = int(cnt_val) if cnt_val.isdigit() else cnt_val
             if cnt_longname not in master_data:
                 master_data[cnt_longname] = {
                     "shortname": cnt_shrtname
                 }

             master_data[cnt_longname][val] = cnt_val

get_links()

#structure of master_data dictionary

"""
master_data:{
  India:{
     metric_1: value,
     metric_2: value,
     .
     .
     metric_n: value
  },
  China:{
     metric_1: value,
     metric_2: value,
     .
     .
     metric_n: value
  },
}
"""

working of `get_links` function:

*   **Iterates through Sources**: It loops through each url (key) and its associated data category (value) in the `source` dictionary.
*   **Fetches Webpage Content**: For every URL, it sends an HTTP GET request to retrieve the webpage's HTML content.
*   **Parses HTML**: It uses BeautifulSoup to parse the fetched HTML, making it easy to extract specific data elements.
*   **Identifies Data Records**: It searches for all `div` elements that have the class `recordsetContainer`. Each of these `div`s is expected to contain data for a specific country.
*   **Extracts Country-Specific Data**: For each identified record container, it extracts three pieces of information:
    *   `cnt_shrtname`: The country's abbreviated name (e.g., "USA").
    *   `cnt_longname`: The full name of the country (e.g., "United States").
    *   `cnt_val`: A numerical value corresponding to the data category for that country. It cleans this value by removing any non-digit characters (like newlines, tabs, "km", commas) and then converts it to an integer. If the cleaned value isn't purely numeric, it defaults to `0`.
*   **Populates `master_data`**: It then organizes this extracted information into the `master_data` dictionary:
    *   If a country's `cnt_longname` is not yet a key in `master_data`, a new entry is created for that country, including its `shortname`.
    *   Finally, the `cnt_val` is added to the respective country's dictionary within `master_data`, using the data category (from the `source` dictionary's `val`) as the key.

In [ ]:
dataframe = pd.DataFrame.from_dict(master_data,orient="index")
dataframe.fillna(0,inplace=True)

dataframe.reset_index(inplace=True)
dataframe.rename(columns={"index":"country"},inplace=True)

In [ ]:
print(dataframe.shape)

(145, 56)


In [ ]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 56 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    145 non-null    object 
 1   shortname                                  145 non-null    object 
 2   total_population                           145 non-null    int64  
 3   total_military_manpower                    145 non-null    int64  
 4   fit_for_service                            145 non-null    int64  
 5   population_reaching_military_age_annually  145 non-null    int64  
 6   active_personnel                           145 non-null    int64  
 7   reserve_personnel                          145 non-null    int64  
 8   paramilitary                               145 non-null    int64  
 9   total_military_aircraft                    145 non-null    int64  
 10  fighter_aircraft          

In [ ]:
print(dataframe.describe())

       total_population  total_military_manpower  fit_for_service  \
count      1.450000e+02             1.450000e+02     1.450000e+02   
mean       5.456547e+07             2.581125e+07     2.019565e+07   
std        1.702420e+08             8.583942e+07     6.928101e+07   
min        3.640360e+05             8.372800e+04     5.023700e+04   
25%        5.650957e+06             2.392390e+06     1.912981e+06   
50%        1.469705e+07             6.351857e+06     4.096295e+06   
75%        3.879481e+07             1.794604e+07     1.403738e+07   
max        1.415043e+09             7.641234e+08     6.268642e+08   

       population_reaching_military_age_annually  active_personnel  \
count                               1.450000e+02      1.450000e+02   
mean                                8.546933e+05      1.538843e+05   
std                                 2.667628e+06      2.968488e+05   
min                                 1.820000e+03      0.000000e+00   
25%                         

In [ ]:
print(dataframe.isna())

     country  shortname  total_population  total_military_manpower  \
0      False      False             False                    False   
1      False      False             False                    False   
2      False      False             False                    False   
3      False      False             False                    False   
4      False      False             False                    False   
..       ...        ...               ...                      ...   
140    False      False             False                    False   
141    False      False             False                    False   
142    False      False             False                    False   
143    False      False             False                    False   
144    False      False             False                    False   

     fit_for_service  population_reaching_military_age_annually  \
0              False                                      False   
1              False     

In [ ]:
dataframe.head()

,country,shortname,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,China,CHN,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,...,225341000000,366160000000,6654000000000,4827000000,5313000000,143197000000,9596960,14500,22457,27700
1,India,IND,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,...,33170000000,58867000000,1381000000000,985671000,1200000000,111052000000,3287263,7000,13888,14500
2,United States,USA,341963408,150463900,124816644,4445524,1328000,799500,0,13043,...,1029000000000,914301000000,13402000000000,548849000,476044000,248941000000,9833517,19924,12002,41009
3,Indonesia,INO,281562465,137965608,114595923,4786562,400000,400000,250000,459,...,57410000000,36061000000,1408000000000,659357000,202283000,34869000000,1904569,54716,2958,21579
4,Pakistan,PAK,252363571,108516336,85803614,4794908,654000,550000,500000,1399,...,36937000000,46448000000,592219000000,12712000,34027000,3064000000,796095,1046,7257,0


In [ ]:
dataframe.to_csv("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/processeddata/unified_military_data.csv",index=False)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/processeddata/unified_military_data.csv")
df.head()

,country,shortname,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,China,CHN,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,...,225341000000,366160000000,6654000000000,4827000000,5313000000,143197000000,9596960,14500,22457,27700
1,India,IND,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,...,33170000000,58867000000,1381000000000,985671000,1200000000,111052000000,3287263,7000,13888,14500
2,United States,USA,341963408,150463900,124816644,4445524,1328000,799500,0,13043,...,1029000000000,914301000000,13402000000000,548849000,476044000,248941000000,9833517,19924,12002,41009
3,Indonesia,INO,281562465,137965608,114595923,4786562,400000,400000,250000,459,...,57410000000,36061000000,1408000000000,659357000,202283000,34869000000,1904569,54716,2958,21579
4,Pakistan,PAK,252363571,108516336,85803614,4794908,654000,550000,500000,1399,...,36937000000,46448000000,592219000000,12712000,34027000,3064000000,796095,1046,7257,0


In [2]:
kpi_df = pd.read_csv("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/processeddata/unified_military_data.csv")

new_kpi_df = kpi_df.copy()

manpow_cols = [
    "total_military_manpower",
    "fit_for_service",
    "population_reaching_military_age_annually",
    "active_personnel",
    "reserve_personnel",
    "paramilitary"
]

air_power = [
    "total_military_aircraft",
    "fighter_aircraft",
    "attack_aircraft",
    "transport_aircraft",
    "trainer_aircraft",
    "special_mission_aircraft",
    "tanker_aircraft",
    "total_military_helicopters",
    "attack_helicopters"
]

land_power = [
    "tanks",
    "armored_fighting_vehicles",
    "self_propelled_artillery",
    "towed_artillery",
    "rocket_projectors"
]

naval_power = [
    "total_naval_fleet",
    "total_naval_fleet_tonnage_mt",
    "aircraft_carriers",
    "helicopter_carriers",
    "submarines",
    "destroyers",
    "frigates",
    "corvettes",
    "coastal_patrol_craft",
    "mine_warfare_craft"
]

logistics_power = [
    "total_serviceable_airports",
    "major_ports_and_terminals",
    "total_merchant_marine_fleet",
    "railway_coverage_km",
    "roadway_coverage_km"
]

energy_power = [
    "oil_production_bbl",
    "oil_consumption_bbl",
    "proven_oil_reserves_bbl",
    "natural_gas_production_cum",
    "natural_gas_consumption_cum",
    "proven_natural_gas_reserves_cum",
    "coal_production_cum",
    "coal_consumption_mt",
    "proven_coal_reserves_cum"
]

economic_power = [
    "defense_budget_usd",
    "labour_force",
    "purchasing_power_parity_usd",
    "foreign_exchange_and_gold_reserves_usd"
]

geographical_power = [
    "total_land_area_sq_km",
    "coastline_coverage_km",
    "border_coverage_km",
    "waterway_coverage_km"
]




#manpower
new_kpi_df["manpower_raw"] = (new_kpi_df[manpow_cols].sum(axis=1))
new_kpi_df["manpower_norm"] = (
    new_kpi_df["manpower_raw"] / new_kpi_df["manpower_raw"].max()
)


#airpower
new_kpi_df["air_power_raw"] = (new_kpi_df[air_power].sum(axis=1))
new_kpi_df["air_power_norm"] = (
    new_kpi_df["air_power_raw"] / new_kpi_df["air_power_raw"].max()
)

#landpower
new_kpi_df["land_power_raw"] = (new_kpi_df[land_power].sum(axis=1))
new_kpi_df["land_power_norm"] = (
    new_kpi_df["land_power_raw"] / new_kpi_df["land_power_raw"].max()
)

#navalpower
new_kpi_df["naval_power_raw"] = (new_kpi_df[naval_power].sum(axis=1))
new_kpi_df["naval_power_norm"] = (
    new_kpi_df["naval_power_raw"] / new_kpi_df["naval_power_raw"].max()
)

#logictics
new_kpi_df["logistics_power_raw"] = (new_kpi_df[logistics_power].sum(axis=1))
new_kpi_df["logistics_power_norm"] = (
    new_kpi_df["logistics_power_raw"] / new_kpi_df["logistics_power_raw"].max()
)

#energy
new_kpi_df["energy_power_raw"] = (new_kpi_df[energy_power].sum(axis=1))
new_kpi_df["energy_power_norm"] = (
    new_kpi_df["energy_power_raw"] / new_kpi_df["energy_power_raw"].max()
)

#geographical
new_kpi_df["geographical_power_raw"] = (new_kpi_df[geographical_power].sum(axis=1))
new_kpi_df["geographical_power_norm"] = (
    new_kpi_df["geographical_power_raw"] / new_kpi_df["geographical_power_raw"].max()
)

#economic
new_kpi_df["economic_power_raw"] = (new_kpi_df[economic_power].sum(axis=1))
new_kpi_df["economic_power_norm"] = (
    new_kpi_df["economic_power_raw"] / new_kpi_df["economic_power_raw"].max()
)


new_kpi_df["debt_ratio"] = new_kpi_df["external_debt_usd"]/new_kpi_df["purchasing_power_parity_usd"]
new_kpi_df["debt_norm"] = 1-(new_kpi_df["debt_ratio"]/new_kpi_df["debt_ratio"].max())

new_kpi_df["budget_to_gdp"] = (
    new_kpi_df["defense_budget_usd"] /
    new_kpi_df["purchasing_power_parity_usd"]
)

new_kpi_df["budget_to_gdp_norm"] = (
    new_kpi_df["budget_to_gdp"] /
    new_kpi_df["budget_to_gdp"].max()
)

assets = [
            "manpower_norm",
            "air_power_norm",
            "land_power_norm",
            "naval_power_norm",
            "logistics_power_norm",
            "energy_power_norm",
            "economic_power_norm"
        ]
new_kpi_df["assets_per_capita"] = new_kpi_df[assets].mean(axis=1)/new_kpi_df["total_population"]
new_kpi_df["assets_per_capita_norm"] = (
    new_kpi_df["assets_per_capita"]
    / new_kpi_df["assets_per_capita"].max()
)



nato_countries = [
    "Albania", "Belgium", "Bulgaria", "Canada", "Croatia", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Italy", "Latvia", "Lithuania", "Luxembourg",
    "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland",
    "Portugal", "Romania", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Turkey", "United Kingdom", "United States"
]
eu_countries = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
    "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia",
    "Slovenia", "Spain", "Sweden"
]
brics_countries = [
    "Brazil", "Russia", "India", "China", "South Africa", "Egypt",
    "Ethiopia", "Iran", "United Arab Emirates"
]


new_kpi_df['is_nato'] = new_kpi_df['country'].isin(nato_countries).astype(int)
new_kpi_df['is_eu'] = new_kpi_df['country'].isin(eu_countries).astype(int)
new_kpi_df['is_brics'] = new_kpi_df['country'].isin(brics_countries).astype(int)


new_kpi_df.drop(columns=manpow_cols+land_power+economic_power+naval_power+logistics_power+energy_power+air_power+geographical_power,inplace=True)



# new_kpi_df[["country", "manpower_norm"]].sort_values(
#     by="manpower_norm", ascending=False
# ).head(5)
new_kpi_df.drop(columns=["budget_to_gdp","assets_per_capita","debt_ratio","shortname","external_debt_usd","total_population","land_power_raw","air_power_raw","manpower_raw","economic_power_raw","geographical_power_raw","energy_power_raw","logistics_power_raw","naval_power_raw"],inplace=True)

new_kpi_df["overall_power_index"] = (
    new_kpi_df[
        [
            "manpower_norm",
            "air_power_norm",
            "land_power_norm",
            "naval_power_norm",
            "logistics_power_norm",
            "energy_power_norm",
            "geographical_power_norm",
            "debt_norm",
            "budget_to_gdp_norm",
            "assets_per_capita_norm",
        ]
    ].mean(axis=1)
)

new_kpi_df["overall_rank"] = (
    new_kpi_df["overall_power_index"]
    .rank(ascending=False, method="dense")
).astype(int)

new_kpi_df["power_rank_gap"] = (
    new_kpi_df["overall_rank"] - new_kpi_df["overall_rank"].min()
)

new_kpi_df.sort_values(by="overall_rank",inplace=True)
new_kpi_df.reset_index(drop=True,inplace=True)

cols = ["overall_rank"] + [c for c in new_kpi_df.columns if c != "overall_rank"]
new_kpi_df = new_kpi_df[cols]

new_kpi_df.head()
# new_kpi_df.describe()
# new_kpi_df.info()


,overall_rank,country,manpower_norm,air_power_norm,land_power_norm,naval_power_norm,logistics_power_norm,energy_power_norm,geographical_power_norm,economic_power_norm,debt_norm,budget_to_gdp_norm,assets_per_capita_norm,is_nato,is_eu,is_brics,overall_power_index,power_rank_gap
0,1,United States,0.199335,1.000000,1.000000,1.000000,1.000000,0.318155,0.573944,0.753495,0.983073,0.222801,0.073590,1,0,0,0.637090,0
1,2,China,1.000000,0.251935,0.396007,0.685665,0.776626,0.151102,0.559759,1.000000,0.999270,0.052464,0.014377,0,0,1,0.488721,1
2,3,Russia,0.084888,0.337752,0.385729,0.302543,0.198944,1.000000,1.000000,0.187133,0.998978,0.133005,0.084656,0,0,1,0.452649,2
3,4,India,0.858697,0.166228,0.393694,0.142514,0.933246,0.032376,0.192502,0.395122,0.999470,0.035138,0.009900,0,0,1,0.376377,3
4,5,Qatar,0.000868,0.019208,0.013021,0.000054,0.001040,0.489888,0.000709,0.010472,0.988523,0.189874,1.000000,0,0,0,0.270318,4


In [ ]:
new_kpi_df.describe()

,overall_rank,manpower_norm,air_power_norm,land_power_norm,naval_power_norm,logistics_power_norm,energy_power_norm,geographical_power_norm,economic_power_norm,debt_norm,budget_to_gdp_norm,assets_per_capita_norm,is_nato,is_eu,is_brics,overall_power_index,power_rank_gap
count,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,1.450000e+02,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000
mean,73.000000,0.033479,0.027487,0.041741,0.026038,0.041490,2.955381e-02,0.053711,0.035736,0.984329,0.087534,0.035290,0.206897,0.165517,0.062069,0.136065,72.000000
std,42.001984,0.111926,0.091438,0.105437,0.105339,0.133229,1.149525e-01,0.127447,0.111320,0.084079,0.112739,0.085829,0.406485,0.372935,0.242117,0.067984,42.001984
min,1.000000,0.000096,0.000000,0.000210,0.000000,0.000355,8.140328e-11,0.000053,0.000165,0.000000,0.004618,0.002610,0.000000,0.000000,0.000000,0.007900,0.000000
25%,37.000000,0.003164,0.002008,0.003245,0.000002,0.003213,2.692865e-05,0.004982,0.001891,0.993134,0.031753,0.009900,0.000000,0.000000,0.000000,0.110392,36.000000
50%,73.000000,0.007886,0.007157,0.011019,0.000012,0.009940,5.602095e-04,0.016452,0.006715,0.996975,0.056058,0.019507,0.000000,0.000000,0.000000,0.117777,72.000000
75%,109.000000,0.023700,0.019793,0.031609,0.011943,0.020786,5.924095e-03,0.045343,0.024462,0.998249,0.098280,0.036880,0.000000,0.000000,0.000000,0.136122,108.000000
max,145.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000e+00,1.000000,1.000000,0.999946,1.000000,1.000000,1.000000,1.000000,1.000000,0.637090,144.000000


In [ ]:
new_kpi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   overall_rank             145 non-null    int64  
 1   country                  145 non-null    object 
 2   manpower_norm            145 non-null    float64
 3   air_power_norm           145 non-null    float64
 4   land_power_norm          145 non-null    float64
 5   naval_power_norm         145 non-null    float64
 6   logistics_power_norm     145 non-null    float64
 7   energy_power_norm        145 non-null    float64
 8   geographical_power_norm  145 non-null    float64
 9   economic_power_norm      145 non-null    float64
 10  debt_norm                145 non-null    float64
 11  budget_to_gdp_norm       145 non-null    float64
 12  assets_per_capita_norm   145 non-null    float64
 13  is_nato                  145 non-null    int64  
 14  is_eu                    1

In [ ]:
new_kpi_df.to_csv("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/processeddata/unified_military_KPI.csv",index=False)